# QuantLib Benchmark: SOFR + TermSOFR3m + CLP ICP Curves & Swap Sensitivities

Replicates the `QuantSupport` sensitivity example using QuantLib-Python.

**Key conventions (matching rustatlas):**
- Reference date: 2025-11-11
- Settlement: T+2 calendar days → 2025-11-13 (NullCalendar)
- Log-linear interpolation on discount factors (flat-forward)
- **NullCalendar** / **Unadjusted** BDC everywhere
- **Backward** date generation, no end-of-month
- Floating leg uses **simple forward rates** per coupon period (NOT daily-compounded OIS)
- Fixed leg: Semiannual, Floating leg: Quarterly
- Day count: Act/360, Compounding: Simple
- Bootstrap instruments start at reference_date (not T+2)
- DV01 via central-difference bump ±1bp

In [113]:
import QuantLib as ql
import pandas as pd
import numpy as np

## 1. Global Settings

In [114]:
reference_date = ql.Date(11, 11, 2025)
ql.Settings.instance().evaluationDate = reference_date

# rustatlas uses NullCalendar + Unadjusted for everything
calendar = ql.NullCalendar()
day_count = ql.Actual360()

# T+2 settlement (calendar days, no holiday adjustment)
start_date = reference_date + 2  # 2025-11-13
notional = 10_000_000.0

print(f"Reference date : {reference_date}")
print(f"Start date     : {start_date}")

Reference date : November 11th, 2025
Start date     : November 13th, 2025


## 2. Market Data

All quotes from `examples/sensitivity/data/quotes.json`.

In [115]:
# SOFR OIS curve quotes
sofr_quotes = {
    "FixedRateDeposit_USD_SOFR_1D": 0.04330,
    "OIS_USD_SOFR_3M":  0.04335,
    "OIS_USD_SOFR_6M":  0.04250,
    "OIS_USD_SOFR_1Y":  0.04100,
    "OIS_USD_SOFR_2Y":  0.03920,
    "OIS_USD_SOFR_3Y":  0.03840,
    "OIS_USD_SOFR_4Y":  0.03800,
    "OIS_USD_SOFR_5Y":  0.03780,
    "OIS_USD_SOFR_7Y":  0.03770,
    "OIS_USD_SOFR_10Y": 0.03790,
    "OIS_USD_SOFR_15Y": 0.03850,
    "OIS_USD_SOFR_20Y": 0.03880,
    "OIS_USD_SOFR_30Y": 0.03750,
}

# TermSOFR3m curve quotes (basis spreads over SOFR)
termsofr_quotes = {
    "FixedRateDeposit_USD_TermSOFR3m_3M": 0.04365,
    "BasisSwap_USD_SOFR_TermSOFR3m_6M":  0.000127237986,
    "BasisSwap_USD_SOFR_TermSOFR3m_1Y":  0.000160745572,
    "BasisSwap_USD_SOFR_TermSOFR3m_2Y":  0.000196298404,
    "BasisSwap_USD_SOFR_TermSOFR3m_3Y":  0.000224136825,
    "BasisSwap_USD_SOFR_TermSOFR3m_4Y":  0.000243421966,
    "BasisSwap_USD_SOFR_TermSOFR3m_5Y":  0.000262792898,
    "BasisSwap_USD_SOFR_TermSOFR3m_7Y":  0.000298893256,
    "BasisSwap_USD_SOFR_TermSOFR3m_10Y": 0.000335114864,
    "BasisSwap_USD_SOFR_TermSOFR3m_15Y": 0.000378100224,
    "BasisSwap_USD_SOFR_TermSOFR3m_20Y": 0.000415419903,
    "BasisSwap_USD_SOFR_TermSOFR3m_30Y": 0.000465137190,
}

# CLP ICP curve quotes
icp_quotes = {
    "FixedRateDeposit_CLP_ICP_1D":  0.0500,
    "FixedRateDeposit_CLP_ICP_3M":  0.0498,
    "OIS_CLP_ICP_6M":  0.0495,
    "OIS_CLP_ICP_1Y":  0.0485,
    "OIS_CLP_ICP_2Y":  0.0470,
    "OIS_CLP_ICP_3Y":  0.0455,
    "OIS_CLP_ICP_5Y":  0.0440,
    "OIS_CLP_ICP_7Y":  0.0430,
    "OIS_CLP_ICP_10Y": 0.0425,
    "OIS_CLP_ICP_20Y": 0.0420,
}

# Merge all quotes into a single handle store for bumping
all_quote_handles = {}
for quotes_dict in [sofr_quotes, termsofr_quotes, icp_quotes]:
    for name, rate in quotes_dict.items():
        all_quote_handles[name] = ql.SimpleQuote(rate)

def make_handle(name):
    return ql.QuoteHandle(all_quote_handles[name])

print(f"Total quotes: {len(all_quote_handles)}")

Total quotes: 35


## 3. Helper Functions

Build swaps matching rustatlas conventions: simple forward rates (VanillaSwap),
NullCalendar, Unadjusted, Backward date generation.

In [116]:
def make_schedule(start, end, frequency):
    """Schedule matching rustatlas: NullCalendar, Unadjusted, Backward."""
    return ql.Schedule(
        start, end,
        frequency,
        calendar,            # NullCalendar
        ql.Unadjusted,       # convention
        ql.Unadjusted,       # termination convention
        ql.DateGeneration.Backward,
        False,               # end of month = false
    )


def make_vanilla_swap(start, maturity, fixed_rate, notional_val,
                      forecast_handle, discount_handle, ibor_name="SimpleSOFR",
                      ccy=None):
    """
    Build a VanillaSwap (Receiver = receive fixed, pay float).
    Uses simple forward rates from the forecast curve (NOT daily-compounded OIS).
    """
    if ccy is None:
        ccy = ql.USDCurrency()
    fixed_schedule = make_schedule(start, maturity, ql.Period(ql.Semiannual))
    float_schedule = make_schedule(start, maturity, ql.Period(ql.Quarterly))

    ibor_index = ql.IborIndex(
        ibor_name, ql.Period(3, ql.Months),
        0, ccy, calendar, ql.Unadjusted, False, day_count,
        forecast_handle,
    )

    swap = ql.VanillaSwap(
        ql.VanillaSwap.Receiver,
        notional_val,
        fixed_schedule,
        fixed_rate,
        day_count,
        float_schedule,
        ibor_index,
        0.0,
        day_count,
    )

    engine = ql.DiscountingSwapEngine(discount_handle)
    swap.setPricingEngine(engine)
    return swap


def compute_dv01(swap, handles_to_bump, fx_divisor=1.0):
    """Central-difference DV01: bump each pillar +/-1bp."""
    bump = 1e-4
    base = swap.NPV() / fx_divisor
    results = {}
    for name, sq in handles_to_bump.items():
        orig = sq.value()
        sq.setValue(orig + bump)
        up = swap.NPV() / fx_divisor
        sq.setValue(orig - bump)
        dn = swap.NPV() / fx_divisor
        sq.setValue(orig)
        results[name] = (up - dn) / 2.0
    return base, results


def display_dv01(label, base_npv, dv01_dict, qs_dict=None, rs_npv=None):
    """Print DV01 table, optionally with rustatlas comparison."""
    print("═" * 75)
    print(f"  {label}")
    npv_str = f"  NPV = {base_npv:>14.2f} USD"
    if rs_npv is not None:
        npv_str += f"  (rustatlas: {rs_npv:>14.2f} USD)"
    print(npv_str)
    print("═" * 75)
    if qs_dict:
        print(f"  {'Pillar':<45} {'QL DV01':>10} {'QS DV01':>10} {'Diff':>10}")
    else:
        print(f"  {'Pillar':<45} {'DV01 (USD/bp)':>16}")
    print(f"  {'-'*73}")
    total_ql = 0.0
    total_rs = 0.0
    for name, dv01 in dv01_dict.items():
        total_ql += dv01
        if qs_dict:
            rs = qs_dict.get(name, 0.0)
            total_rs += rs
            print(f"  {name:<45} {dv01:>10.2f} {rs:>10.2f} {dv01-rs:>10.2f}")
        else:
            print(f"  {name:<45} {dv01:>16.2f}")
    print(f"  {'-'*73}")
    if qs_dict:
        print(f"  {'TOTAL':<45} {total_ql:>10.2f} {total_rs:>10.2f} {total_ql-total_rs:>10.2f}")
    else:
        print(f"  {'TOTAL':<45} {total_ql:>16.2f}")
    print()

## 4. Bootstrap the SOFR OIS Curve

rustatlas bootstraps OIS swaps with:
- Fixed Semi / Float Quarterly, both starting at reference_date
- Simple forward rates (not daily-compounded OIS)
- NPV = 0 as the bootstrap objective

We use `SwapRateHelper` with a quarterly `IborIndex` (simple forwards), not `OISRateHelper`.

In [117]:
sofr_helpers = []

# 1D deposit
sofr_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_USD_SOFR_1D"),
        ql.Period(1, ql.Days),
        0,                    # fixing days = 0 (starts at ref date)
        calendar,             # NullCalendar
        ql.Unadjusted,
        False,
        day_count,
    )
)

# IborIndex for projecting simple forwards in bootstrap
sofr_ibor_for_bootstrap = ql.IborIndex(
    "SimpleSOFR", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
)

ois_tenors = [
    ("OIS_USD_SOFR_3M",  ql.Period(3, ql.Months)),
    ("OIS_USD_SOFR_6M",  ql.Period(6, ql.Months)),
    ("OIS_USD_SOFR_1Y",  ql.Period(1, ql.Years)),
    ("OIS_USD_SOFR_2Y",  ql.Period(2, ql.Years)),
    ("OIS_USD_SOFR_3Y",  ql.Period(3, ql.Years)),
    ("OIS_USD_SOFR_4Y",  ql.Period(4, ql.Years)),
    ("OIS_USD_SOFR_5Y",  ql.Period(5, ql.Years)),
    ("OIS_USD_SOFR_7Y",  ql.Period(7, ql.Years)),
    ("OIS_USD_SOFR_10Y", ql.Period(10, ql.Years)),
    ("OIS_USD_SOFR_15Y", ql.Period(15, ql.Years)),
    ("OIS_USD_SOFR_20Y", ql.Period(20, ql.Years)),
    ("OIS_USD_SOFR_30Y", ql.Period(30, ql.Years)),
]

for name, tenor in ois_tenors:
    sofr_helpers.append(
        ql.SwapRateHelper(
            make_handle(name),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            sofr_ibor_for_bootstrap,
        )
    )

sofr_curve = ql.PiecewiseLogLinearDiscount(reference_date, sofr_helpers, day_count)
sofr_curve.enableExtrapolation()
sofr_curve_handle = ql.YieldTermStructureHandle(sofr_curve)

print("SOFR curve bootstrapped.")
for dt, df in sofr_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

SOFR curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  November 12th, 2025  DF=0.9998797367  YF=0.002778
  February 11th, 2026  DF=0.9890430514  YF=0.255556
  May 11th, 2026  DF=0.9790789858  YF=0.502778
  November 11th, 2026  DF=0.9597061980  YF=1.013889
  November 11th, 2027  DF=0.9243882594  YF=2.027778
  November 11th, 2028  DF=0.8908288784  YF=3.044444
  November 11th, 2029  DF=0.8585791637  YF=4.058333
  November 11th, 2030  DF=0.8273197252  YF=5.072222
  November 11th, 2032  DF=0.7673403029  YF=7.102778
  November 11th, 2035  DF=0.6833842370  YF=10.144444
  November 11th, 2040  DF=0.5586557094  YF=15.219444
  November 11th, 2045  DF=0.4566471726  YF=20.291667
  November 11th, 2055  DF=0.3281415350  YF=30.436111


## 5. Price SOFR 5Y Swap & Sensitivities

In [118]:
maturity_5y = start_date + ql.Period(5, ql.Years)
print(f"Swap: {start_date} → {maturity_5y}, fixed = 3.78%, notional = {notional:,.0f}")

sofr_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.0378, notional,
    sofr_curve_handle, sofr_curve_handle,
)

rs_sofr_dv01 = {
    "FixedRateDeposit_USD_SOFR_1D": 2.75,
    "OIS_USD_SOFR_3M":  2.78,
    "OIS_USD_SOFR_6M":  0.10,
    "OIS_USD_SOFR_1Y":  0.00,
    "OIS_USD_SOFR_2Y":  -0.00,
    "OIS_USD_SOFR_3Y":  -0.00,
    "OIS_USD_SOFR_4Y":  -0.00,
    "OIS_USD_SOFR_5Y": -4555.18,
    "OIS_USD_SOFR_7Y":   -17.69,
    "OIS_USD_SOFR_10Y":   0.00,
    "OIS_USD_SOFR_15Y":   0.00,
    "OIS_USD_SOFR_20Y":   0.00,
    "OIS_USD_SOFR_30Y":   0.00,
}

sofr_handles = {k: all_quote_handles[k] for k in sofr_quotes}
sofr_npv, sofr_dv01 = compute_dv01(sofr_swap, sofr_handles)
display_dv01("SOFR OIS 5Y Swap", sofr_npv, sofr_dv01, rs_sofr_dv01, rs_npv=342.60)

Swap: November 13th, 2025 → November 13th, 2030, fixed = 3.78%, notional = 10,000,000
═══════════════════════════════════════════════════════════════════════════
  SOFR OIS 5Y Swap
  NPV =         342.60 USD  (rustatlas:         342.60 USD)
═══════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01       Diff
  -------------------------------------------------------------------------
  FixedRateDeposit_USD_SOFR_1D                        2.75       2.75      -0.00
  OIS_USD_SOFR_3M                                     2.78       2.78      -0.00
  OIS_USD_SOFR_6M                                     0.10       0.10       0.00
  OIS_USD_SOFR_1Y                                     0.00       0.00       0.00
  OIS_USD_SOFR_2Y                                    -0.00      -0.00      -0.00
  OIS_USD_SOFR_3Y                                    -0.00      -0.00      -0.00
  OIS_USD_SOFR_4Y                       

## 6. Bootstrap TermSOFR3m Curve

The TermSOFR3m curve is built from:
- A 3M deposit (outright rate)
- Basis swaps: pay SOFR + spread, receive TermSOFR3m Q → NPV = 0

The basis spread quote means `TermSOFR3m_par_swap_rate ≈ SOFR_par_swap_rate + spread`.

In [119]:
termsofr_helpers = []

# 3M deposit
termsofr_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_USD_TermSOFR3m_3M"),
        ql.Period(3, ql.Months),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

# For each basis swap tenor, compute SOFR par rate + spread → TermSOFR3m par rate
sofr_ibor_linked = ql.IborIndex(
    "SimpleSOFR", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
    sofr_curve_handle,
)

termsofr_ibor_for_bootstrap = ql.IborIndex(
    "TermSOFR3m", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
)

basis_tenors = [
    ("BasisSwap_USD_SOFR_TermSOFR3m_6M",  ql.Period(6, ql.Months)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_1Y",  ql.Period(1, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_2Y",  ql.Period(2, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_3Y",  ql.Period(3, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_4Y",  ql.Period(4, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_5Y",  ql.Period(5, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_7Y",  ql.Period(7, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_10Y", ql.Period(10, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_15Y", ql.Period(15, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_20Y", ql.Period(20, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_30Y", ql.Period(30, ql.Years)),
]

# Store par rate SimpleQuote objects so bumping basis spreads propagates
# to the TermSOFR3m curve during DV01 computation.
termsofr_par_quotes = {}

# Annuity ratios: the basis spread enters the quarterly floating leg,
# but SwapRateHelper uses a semiannual fixed-rate swap. The exact
# conversion is:  r_TermSOFR = r_SOFR + spread × A_qtr / A_semi
# where A_qtr / A_semi is the ratio of floating-to-fixed leg annuities.
annuity_ratios = {}

for name, tenor in basis_tenors:
    maturity = reference_date + tenor
    fixed_sched = make_schedule(reference_date, maturity, ql.Period(ql.Semiannual))
    float_sched = make_schedule(reference_date, maturity, ql.Period(ql.Quarterly))

    # Get SOFR par rate for this tenor
    temp_swap = ql.VanillaSwap(
        ql.VanillaSwap.Receiver, 1.0,
        fixed_sched, 0.0, day_count,
        float_sched, sofr_ibor_linked, 0.0, day_count,
    )
    temp_swap.setPricingEngine(ql.DiscountingSwapEngine(sofr_curve_handle))
    sofr_par = temp_swap.fairRate()

    # Annuity ratio: floating-leg BPS / fixed-leg BPS
    # BPS = Σ τ_i × N × DF_i × 1e-4, so the ratio cancels units.
    ratio = abs(temp_swap.floatingLegBPS()) / abs(temp_swap.fixedLegBPS())
    annuity_ratios[name] = ratio

    basis_spread = all_quote_handles[name].value()
    termsofr_par = sofr_par + basis_spread * ratio

    par_sq = ql.SimpleQuote(termsofr_par)
    termsofr_par_quotes[name] = par_sq

    termsofr_helpers.append(
        ql.SwapRateHelper(
            ql.QuoteHandle(par_sq),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            termsofr_ibor_for_bootstrap,
        )
    )

termsofr_curve = ql.PiecewiseLogLinearDiscount(reference_date, termsofr_helpers, day_count)
termsofr_curve.enableExtrapolation()
termsofr_curve_handle = ql.YieldTermStructureHandle(termsofr_curve)

print("TermSOFR3m curve bootstrapped.")
for dt, df in termsofr_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")
print()
print("Annuity ratios (A_qtr / A_semi):")
for name, r in annuity_ratios.items():
    print(f"  {name}: {r:.6f}")

TermSOFR3m curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  February 11th, 2026  DF=0.9889680613  YF=0.255556
  May 11th, 2026  DF=0.9790173488  YF=0.502778
  November 11th, 2026  DF=0.9595519369  YF=1.013889
  November 11th, 2027  DF=0.9240245706  YF=2.027778
  November 11th, 2028  DF=0.8902264182  YF=3.044444
  November 11th, 2029  DF=0.8577367613  YF=4.058333
  November 11th, 2030  DF=0.8262210133  YF=5.072222
  November 11th, 2032  DF=0.7657054656  YF=7.102778
  November 11th, 2035  DF=0.6810318318  YF=10.144444
  November 11th, 2040  DF=0.5553449820  YF=15.219444
  November 11th, 2045  DF=0.4526013223  YF=20.291667
  November 11th, 2055  DF=0.3232702248  YF=30.436111

Annuity ratios (A_qtr / A_semi):
  BasisSwap_USD_SOFR_TermSOFR3m_6M: 1.005173
  BasisSwap_USD_SOFR_TermSOFR3m_1Y: 1.005097
  BasisSwap_USD_SOFR_TermSOFR3m_2Y: 1.004906
  BasisSwap_USD_SOFR_TermSOFR3m_3Y: 1.004822
  BasisSwap_USD_SOFR_TermSOFR3m_4Y: 1.004776
  BasisSwap_USD_SOFR_TermSOFR3m_5Y:

## 7. Price TermSOFR3m 5Y Swap & Sensitivities

Receive fixed 4.00% semi vs pay TermSOFR3m quarterly, 10MM USD.
Discount with SOFR (CSA = SOFR/USD).

In [120]:
termsofr_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.04, notional,
    termsofr_curve_handle, sofr_curve_handle,  # forecast=TermSOFR3m, discount=SOFR
    ibor_name="TermSOFR3m",
)

print(f"TermSOFR3m swap: {start_date} → {maturity_5y}, fixed = 4.00%")
print(f"NPV = {termsofr_swap.NPV():,.2f} USD")
print(f"Fair rate = {termsofr_swap.fairRate():.6%}")

TermSOFR3m swap: November 13th, 2025 → November 13th, 2030, fixed = 4.00%
NPV = 88,828.25 USD
Fair rate = 3.805514%


In [121]:
rs_termsofr_dv01 = {
    "FixedRateDeposit_USD_TermSOFR3m_3M":  5.56,
    "BasisSwap_USD_SOFR_TermSOFR3m_6M":   -0.01,
    "BasisSwap_USD_SOFR_TermSOFR3m_1Y":    0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_2Y":   -0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_3Y":   -0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_4Y":   -0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_5Y": -4576.89,
    "BasisSwap_USD_SOFR_TermSOFR3m_7Y":  -17.69,
    "BasisSwap_USD_SOFR_TermSOFR3m_10Y":    0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_15Y":    0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_20Y":    0.00,
    "BasisSwap_USD_SOFR_TermSOFR3m_30Y":    0.00,
    "FixedRateDeposit_USD_SOFR_1D":        0.00,
    "OIS_USD_SOFR_3M":                     2.72,
    "OIS_USD_SOFR_6M":                     -2.13,
    "OIS_USD_SOFR_1Y":                     -0.76,
    "OIS_USD_SOFR_2Y":                     -4.44,
    "OIS_USD_SOFR_3Y":                     -7.92,
    "OIS_USD_SOFR_4Y":                    -10.30,
    "OIS_USD_SOFR_5Y":                    -30.45,
    "OIS_USD_SOFR_7Y":                     -0.19,
    "OIS_USD_SOFR_10Y":                     0.00,
    "OIS_USD_SOFR_15Y":                     0.00,
    "OIS_USD_SOFR_20Y":                     0.00,
    "OIS_USD_SOFR_30Y":                     0.00,
}

# Custom DV01 for TermSOFR3m: when bumping a basis spread quote by Δ,
# the par rate changes by Δ × (A_qtr / A_semi) — the annuity ratio
# accounts for the spread entering the quarterly leg but the par rate
# being a semiannual fixed rate.
def compute_termsofr_dv01(swap, handles_to_bump, par_quotes, a_ratios):
    bump = 1e-4
    base = swap.NPV()
    results = {}
    for name, sq in handles_to_bump.items():
        orig = sq.value()
        par_sq = par_quotes.get(name)
        par_orig = par_sq.value() if par_sq else None
        ratio = a_ratios.get(name, 1.0)

        sq.setValue(orig + bump)
        if par_sq:
            par_sq.setValue(par_orig + bump * ratio)
        up = swap.NPV()

        sq.setValue(orig - bump)
        if par_sq:
            par_sq.setValue(par_orig - bump * ratio)
        dn = swap.NPV()

        sq.setValue(orig)
        if par_sq:
            par_sq.setValue(par_orig)
        results[name] = (up - dn) / 2.0
    return base, results

combined_handles = {}
for k in termsofr_quotes:
    combined_handles[k] = all_quote_handles[k]
for k in sofr_quotes:
    combined_handles[k] = all_quote_handles[k]

termsofr_npv, termsofr_dv01 = compute_termsofr_dv01(
    termsofr_swap, combined_handles, termsofr_par_quotes, annuity_ratios,
)
display_dv01("TermSOFR3m 5Y Swap", termsofr_npv, termsofr_dv01, rs_termsofr_dv01,
             rs_npv=88754.68)

═══════════════════════════════════════════════════════════════════════════
  TermSOFR3m 5Y Swap
  NPV =       88828.25 USD  (rustatlas:       88754.68 USD)
═══════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01       Diff
  -------------------------------------------------------------------------
  FixedRateDeposit_USD_TermSOFR3m_3M                  2.77       5.56      -2.79
  BasisSwap_USD_SOFR_TermSOFR3m_6M                    1.88      -0.01       1.89
  BasisSwap_USD_SOFR_TermSOFR3m_1Y                   -0.30       0.00      -0.30
  BasisSwap_USD_SOFR_TermSOFR3m_2Y                    1.65      -0.00       1.65
  BasisSwap_USD_SOFR_TermSOFR3m_3Y                    3.69      -0.00       3.69
  BasisSwap_USD_SOFR_TermSOFR3m_4Y                    4.66      -0.00       4.66
  BasisSwap_USD_SOFR_TermSOFR3m_5Y                -4555.51   -4576.89      21.38
  BasisSwap_USD_SOFR_TermSOFR3m_7Y         

## 8. Bootstrap CLP ICP Curve

Same conventions: NullCalendar, Unadjusted, Backward, Semi fixed / Quarterly float.

In [122]:
icp_helpers = []

# 1D deposit
icp_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_1D"),
        ql.Period(1, ql.Days),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

# 3M deposit
icp_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_3M"),
        ql.Period(3, ql.Months),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

icp_ibor_for_bootstrap = ql.IborIndex(
    "SimpleICP", ql.Period(3, ql.Months),
    0, ql.CLPCurrency(), calendar, ql.Unadjusted, False, day_count,
)

icp_tenors = [
    ("OIS_CLP_ICP_6M",  ql.Period(6, ql.Months)),
    ("OIS_CLP_ICP_1Y",  ql.Period(1, ql.Years)),
    ("OIS_CLP_ICP_2Y",  ql.Period(2, ql.Years)),
    ("OIS_CLP_ICP_3Y",  ql.Period(3, ql.Years)),
    ("OIS_CLP_ICP_5Y",  ql.Period(5, ql.Years)),
    ("OIS_CLP_ICP_7Y",  ql.Period(7, ql.Years)),
    ("OIS_CLP_ICP_10Y", ql.Period(10, ql.Years)),
    ("OIS_CLP_ICP_20Y", ql.Period(20, ql.Years)),
]

for name, tenor in icp_tenors:
    icp_helpers.append(
        ql.SwapRateHelper(
            make_handle(name),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            icp_ibor_for_bootstrap,
        )
    )

icp_curve = ql.PiecewiseLogLinearDiscount(reference_date, icp_helpers, day_count)
icp_curve.enableExtrapolation()
icp_curve_handle = ql.YieldTermStructureHandle(icp_curve)

print("CLP ICP curve bootstrapped.")
for dt, df in icp_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

CLP ICP curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  November 12th, 2025  DF=0.9998611304  YF=0.002778
  February 11th, 2026  DF=0.9874332660  YF=0.255556
  May 11th, 2026  DF=0.9757168470  YF=0.502778
  November 11th, 2026  DF=0.9525936769  YF=1.013889
  November 11th, 2027  DF=0.9101874364  YF=2.027778
  November 11th, 2028  DF=0.8722701847  YF=3.044444
  November 11th, 2030  DF=0.8025817025  YF=5.072222
  November 11th, 2032  DF=0.7404184262  YF=7.102778
  November 11th, 2035  DF=0.6543257930  YF=10.144444
  November 11th, 2045  DF=0.4324258301  YF=20.291667


## 9. Price CLP ICP 5Y Swap & Sensitivities

Receive fixed 4.40% semi vs pay ICP quarterly, 5B CLP.

**Discount curve**: In rustatlas, CLP cashflows are discounted off `Collateral(CLP, USD)` —
a cross-currency basis curve built from FX forwards + XCcy swaps.
Here we use the ICP curve itself (self-discount) as an approximation,
so NPVs and DV01s will differ from rustatlas.
NPVs converted to USD at FX spot = 935.

In [123]:
clp_notional = 5_000_000_000.0
fx_spot = 935.0

icp_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.0440, clp_notional,
    icp_curve_handle, icp_curve_handle,  # self-discount
    ibor_name="SimpleICP", ccy=ql.CLPCurrency(),
)

icp_npv_clp = icp_swap.NPV()
icp_npv_usd = icp_npv_clp / fx_spot

print(f"CLP ICP swap: {start_date} → {maturity_5y}, fixed = 4.40%")
print(f"NPV = {icp_npv_clp:,.2f} CLP = {icp_npv_usd:,.2f} USD")
print(f"Fair rate = {icp_swap.fairRate():.6%}")

CLP ICP swap: November 13th, 2025 → November 13th, 2030, fixed = 4.40%
NPV = 263,033.98 CLP = 281.32 USD
Fair rate = 4.398827%


In [124]:
rs_icp_dv01 = {
    "FixedRateDeposit_CLP_ICP_1D":  1.46,
    "FixedRateDeposit_CLP_ICP_3M":  1.56,
    "OIS_CLP_ICP_6M":              -0.07,
    "OIS_CLP_ICP_1Y":               0.03,
    "OIS_CLP_ICP_2Y":              -0.00,
    "OIS_CLP_ICP_3Y":              -0.00,
    "OIS_CLP_ICP_5Y":           -2381.76,
    "OIS_CLP_ICP_7Y":              -9.22,
    "OIS_CLP_ICP_10Y":              0.00,
    "OIS_CLP_ICP_20Y":              0.00,
}

icp_handles = {k: all_quote_handles[k] for k in icp_quotes}
icp_npv_usd_base, icp_dv01 = compute_dv01(icp_swap, icp_handles, fx_divisor=fx_spot)

print("Note: rustatlas discounts CLP flows with Collateral(CLP,USD) curve;")
print("      here we use ICP self-discount, so some difference is expected.\n")
display_dv01("CLP ICP OIS 5Y Swap (USD equiv)", icp_npv_usd_base, icp_dv01, rs_icp_dv01,
             rs_npv=288.27)

Note: rustatlas discounts CLP flows with Collateral(CLP,USD) curve;
      here we use ICP self-discount, so some difference is expected.

═══════════════════════════════════════════════════════════════════════════
  CLP ICP OIS 5Y Swap (USD equiv)
  NPV =         281.32 USD  (rustatlas:         288.27 USD)
═══════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01       Diff
  -------------------------------------------------------------------------
  FixedRateDeposit_CLP_ICP_1D                         1.47       1.46       0.01
  FixedRateDeposit_CLP_ICP_3M                         1.48       1.56      -0.08
  OIS_CLP_ICP_6M                                      0.06      -0.07       0.13
  OIS_CLP_ICP_1Y                                      0.00       0.03      -0.03
  OIS_CLP_ICP_2Y                                     -0.00      -0.00      -0.00
  OIS_CLP_ICP_3Y                                     -

## 10. Summary

In [125]:
print("NPV Comparison Summary")
print("=" * 70)
print(f"{'Product':<40} {'QuantLib':>12} {'rustatlas':>12}")
print("-" * 70)
print(f"{'SOFR OIS 5Y Swap (USD)':<40} {sofr_npv:>12.2f} {342.60:>12.2f}")
print(f"{'TermSOFR3m 5Y Swap (USD)':<40} {termsofr_npv:>12.2f} {88754.68:>12.2f}")
print(f"{'CLP ICP 5Y Swap (USD equiv)':<40} {icp_npv_usd_base:>12.2f} {288.27:>12.2f}")
print("=" * 70)
print()
print("Convention notes:")
print("  - VanillaSwap with IborIndex (simple fwd rates), not OvernightIndexedSwap")
print("  - NullCalendar, Unadjusted BDC, Backward date generation")
print("  - Log-linear on DFs (flat forward between nodes)")
print("  - TermSOFR3m: static par-rate approach for basis curve")
print("  - CLP ICP: self-discount (missing Collateral(CLP,USD) curve)")

NPV Comparison Summary
Product                                      QuantLib    rustatlas
----------------------------------------------------------------------
SOFR OIS 5Y Swap (USD)                         342.60       342.60
TermSOFR3m 5Y Swap (USD)                     88828.25     88754.68
CLP ICP 5Y Swap (USD equiv)                    281.32       288.27

Convention notes:
  - VanillaSwap with IborIndex (simple fwd rates), not OvernightIndexedSwap
  - NullCalendar, Unadjusted BDC, Backward date generation
  - Log-linear on DFs (flat forward between nodes)
  - TermSOFR3m: static par-rate approach for basis curve
  - CLP ICP: self-discount (missing Collateral(CLP,USD) curve)
